In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

df = pd.read_csv('heart.csv')

# ---------------------------------------------------------------------------
# STEP 1: Distance function
# ---------------------------------------------------------------------------
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))


# ---------------------------------------------------------------------------
# STEP 2: Feature scaling (z-score standardization)
# ---------------------------------------------------------------------------
def fit_scaler(X_train):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return mean, std


def apply_scaler(X, mean, std):
    return (X - mean) / std



# ---------------------------------------------------------------------------
# STEP 3: Predict a single point
# ---------------------------------------------------------------------------
def predict_one(X_train, y_train, X_newValues, k):
    distances = np.array([euclidean_distance(X_newValues, x_train_i) for x_train_i in X_train]) #distance between new value and all training values
    k_nearest_indices = np.argsort(distances)[:k] # find the index of sorted element
    k_nearest_labels = y_train[k_nearest_indices] # find nearest labels like [0, 1, 0, 0, 1]
    most_common = Counter(k_nearest_labels).most_common(1) # [(label: 0, count: 3)]
    return most_common[0][0]


def knn_predict(X_train, y_train, X_newValues, k):
    return np.array([predict_one(X_train, y_train, x, k) for x in X_newValues])


# ---------------------------------------------------------------------------
# STEP 4: Find the best k using K-Fold Cross Validation
# ---------------------------------------------------------------------------
def find_best_k(X_train, y_train, k_values=range(1, 8), n_folds=5):
    indices = np.arange(len(X_train))
    np.random.shuffle(indices)

    folds = np.array_split(indices, n_folds)

    best_k = None
    best_accuracy = 0

    for k in k_values:
        fold_accuracies = []

        for i in range(n_folds):
            val_idx = folds[i]
            train_idx = np.concatenate([folds[j] for j in range(n_folds) if j != i])

            X_tr = X_train[train_idx]
            y_tr = y_train[train_idx]

            X_val = X_train[val_idx]
            y_val = y_train[val_idx]

            mean, std = fit_scaler(X_tr)
            X_tr_scaled = apply_scaler(X_tr, mean, std)
            X_val_scaled = apply_scaler(X_val, mean, std)

            predictions = knn_predict(X_tr_scaled, y_tr, X_val_scaled, k)
            fold_accuracies.append(accuracy(y_val, predictions))

        avg_accuracy = np.mean(fold_accuracies)
        print(f"k = {k:2d}, CV Accuracy = {avg_accuracy:.4f}")

        if avg_accuracy > best_accuracy:
            best_accuracy = avg_accuracy
            best_k = k

    print(f"\nBest k = {best_k} (CV Accuracy = {best_accuracy:.4f})")
    return best_k

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)


# ---------------------------------------------------------------------------
# EXAMPLE USAGE: visualize the data, THEN predict
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # toy data: [age, cholesterol]
    sex_map            = {'M': 1, 'F': 0}
    ExerciseAngina_map = {'Y': 1, 'N': 0}
    ST_slope_map       = {'Up': 1, 'Flat': 0, 'Down': -1}

    df['Sex']            = df['Sex'].map(sex_map)
    df['ExerciseAngina'] = df['ExerciseAngina'].map(ExerciseAngina_map)
    df['ST_Slope']       = df['ST_Slope'].map(ST_slope_map)

    df = pd.get_dummies(df, columns=['ChestPainType'], prefix='ChestPainType', dtype=int)
    df = pd.get_dummies(df, columns=['RestingECG'],    prefix='RestingECG',    dtype=int) 


    X = df.drop('HeartDisease', axis=1).values
    Y = df['HeartDisease'].values
    
    indexes = np.random.permutation(len(X))
    X = X[indexes]
    Y = Y[indexes]
    
    split_index = int(len(X) * 0.8)

    X_train, y_train = X[:split_index], Y[:split_index]
    X_newValues, y_val = X[split_index:], Y[split_index:]

    # fit scaler on TRAIN only, apply to both
    mean, std = fit_scaler(X_train)
    X_train_scaled = apply_scaler(X_train, mean, std)
    X_newValues_scaled = apply_scaler(X_newValues, mean, std)

    # best_k = find_best_k(X_train_scaled, y_train)
    k = 8
    print(k)

    # --- THEN PREDICT ---
    predictions = knn_predict(X_train_scaled, y_train, X_newValues_scaled, k)

    print("Predictions:", predictions)
    print("True labels:", y_val)
    print("Accuracy:", accuracy(y_val, predictions) * 100, "%")



8
Predictions: [0 1 0 0 1 0 1 1 0 1 0 0 1 1 0 1 0 0 0 1 1 1 0 0 1 0 0 1 1 0 1 0 1 0 0 0 1
 0 1 1 1 1 0 1 1 0 1 0 1 0 1 0 0 1 1 1 1 1 1 0 0 0 1 0 1 1 0 0 0 0 1 1 1 1
 1 0 1 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 1 1 1 0 1 0 1 0 1 0 1 0 1 0 1 1 1 0 0
 0 1 0 1 1 0 0 1 0 1 1 1 0 1 1 1 0 0 0 1 1 1 1 1 0 1 1 0 0 1 0 1 1 0 0 1 1
 0 0 0 0 1 0 0 1 1 0 0 1 1 0 0 1 0 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 1 1]
True labels: [0 1 0 1 1 0 1 1 1 1 0 0 1 1 0 1 0 1 0 1 1 1 0 0 1 1 1 1 1 0 1 0 1 0 1 1 1
 0 1 1 1 1 0 1 1 0 1 0 1 0 1 0 0 1 1 1 1 1 1 1 0 0 1 1 1 1 0 0 0 1 1 1 1 1
 1 0 1 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 1 1 1 0 1 0 1 0 1 1 1 0 1 0 1 1 1 0 0
 0 1 0 1 0 0 0 1 0 1 1 1 0 1 1 1 0 0 0 1 1 1 1 1 0 1 1 0 0 0 0 1 0 0 0 1 1
 0 0 0 0 1 0 0 1 0 0 0 1 1 0 0 0 0 0 1 1 0 0 0 1 0 0 1 0 1 1 1 0 1 1 1 1]
Accuracy: 91.30434782608695 %
